# 区域基本面 重构

## 元数据
- 原始路径: 代码库/模版/区域基本面
- 创建时间: 2026-02-15
- 任务ID: T40

---

## 1. 项目概述

### 1.1 功能描述

区域基本面项目专注于分析不同区域的经济基本面，包括GDP增速、财政收入、产业结构、人口结构、投资情况等，为区域债投资和区域风险评估提供基本面数据支持。

**核心功能**：
1. 区域经济数据采集与处理
2. 综合财力计算
3. 经济活力与发展潜力评估
4. 区域综合评分与风险评级
5. 区域对比分析

### 1.2 数据源

- 数据库表：`bond.fiscal_region_city_county`
- API接口：企业预警通/评级狗 区域数据API

### 1.3 输出结果

- 区域综合评分
- 区域风险评级（AAA/AA+/AA/AA-/A/BBB/BB）
- 区域对比排名报告

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import json
import datetime
from pathlib import Path

# 数据处理
import pandas as pd
import numpy as np

# 数据库
import sqlalchemy
from sqlalchemy import text
import pymysql

# 可视化
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False

# 配置
from config import *

print(f"Pandas version: {pd.__version__}")
print(f"SQLAlchemy version: {sqlalchemy.__version__}")
print("环境配置完成!")

### 2.2 配置参数

In [ ]:
# 数据库连接
DB_CONFIG = {
    'host': DB_HOST,
    'port': DB_PORT,
    'user': DB_USER,
    'password': DB_PASSWORD,
    'database': DB_NAME,
    'charset': 'utf8mb4'
}

# 区域配置参数
REGION_CONFIG = {
    'fiscal_weights': {
        'general_budget': 0.7,
        'gov_fund': 0.3
    },
    'debt_warning_levels': {
        'debt_to_gdp': 0.6,
        'debt_to_fiscal': 2.0,
        'debt_growth': 0.2
    },
    'growth_targets': {
        'high_growth': 8.0,
        'medium_growth': 6.0,
        'low_growth': 4.0
    }
}

# 城投债务估算系数（当数据缺失时使用）
LGD_DEBT_MULTIPLIER = 2.0

print("配置参数加载完成!")
print(f"财政权重: 一般预算 {REGION_CONFIG['fiscal_weights']['general_budget']}, 政府基金 {REGION_CONFIG['fiscal_weights']['gov_fund']}")

## 3. 数据获取

### 3.1 数据源连接

In [ ]:
def create_db_engine(config):
    """创建数据库连接引擎"""
    connection_string = f"mysql+pymysql://{config['user']}:{config['password']}@{config['host']}:{config['port']}/{config['database']}"
    engine = sqlalchemy.create_engine(
        connection_string,
        pool_recycle=3600,
        pool_pre_ping=True,
        echo=False
    )
    return engine

# 创建数据库连接
engine = create_db_engine(DB_CONFIG)

# 测试连接
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("数据库连接成功!")
except Exception as e:
    print(f"数据库连接失败: {e}")

### 3.2 数据读取

In [ ]:
def load_region_data(engine, limit=None):
    """加载区域基本面数据"""
    sql = """
    SELECT 
        `DT` as report_date,
        `省` as province,
        `市` as city,
        `区县` as district,
        `地区级别` as region_level,
        `城投平台有息债务余额` as lgd_debt_balance,
        `城投债券余额` as lgd_bond_balance,
        `地方债务余额` as local_debt_balance
    FROM bond.fiscal_region_city_county
    """
    
    if limit:
        sql += f" LIMIT {limit}"
    
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    
    return df

# 加载数据（开发时可使用limit限制数据量）
df_region = load_region_data(engine, limit=1000)

print(f"加载区域数据: {len(df_region)} 条")
print(f"数据列: {df_region.columns.tolist()}")
df_region.head()

## 4. 数据处理

### 4.1 数据清洗

In [ ]:
def clean_region_data(df):
    """清洗区域数据"""
    df_clean = df.copy()
    
    # 转换数值列
    numeric_cols = ['lgd_debt_balance', 'lgd_bond_balance', 'local_debt_balance']
    for col in numeric_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)
    
    # 转换日期
    df_clean['report_date'] = pd.to_datetime(df_clean['report_date'], errors='coerce')
    
    # 填充地区级别
    def get_region_level(row):
        if pd.notna(row['region_level']) and row['region_level'] != '':
            return row['region_level']
        if pd.notna(row['district']) and row['district'] != '':
            return '区县级'
        elif pd.notna(row['city']) and row['city'] != '':
            return '地市级'
        else:
            return '省级'
    
    df_clean['region_level'] = df_clean.apply(get_region_level, axis=1)
    
    return df_clean

# 执行数据清洗
df_clean = clean_region_data(df_region)

print("数据清洗完成!")
print(f"地区级别分布:\n{df_clean['region_level'].value_counts()}")

### 4.2 数据转换 - 债务数据修正

In [ ]:
def correct_debt_data(df):
    """
    修正债务数据
    
    修正逻辑:
    1. 债务余额 < 债券余额 -> 用债券余额填充
    2. 缺失数据 -> 用地方债务余额的2倍估算
    """
    df_corrected = df.copy()
    
    # 修正1: 债务余额 < 债券余额
    mask = df_corrected['lgd_debt_balance'] < df_corrected['lgd_bond_balance']
    count1 = mask.sum()
    df_corrected.loc[mask, 'lgd_debt_balance'] = df_corrected.loc[mask, 'lgd_bond_balance']
    
    # 修正2: 缺失数据用地方债务余额的2倍估算
    mask2 = (df_corrected['lgd_debt_balance'] == 0) | (df_corrected['lgd_debt_balance'].isna())
    count2 = mask2.sum()
    df_corrected.loc[mask2, 'lgd_debt_balance'] = df_corrected.loc[mask2, 'local_debt_balance'] * LGD_DEBT_MULTIPLIER
    
    return df_corrected, {'debt_lt_bond': count1, 'missing_filled': count2}

# 执行债务数据修正
df_corrected, correction_stats = correct_debt_data(df_clean)

print(f"债务数据修正完成!")
print(f"  - 债务<债券修正: {correction_stats['debt_lt_bond']} 条")
print(f"  - 缺失数据填充: {correction_stats['missing_filled']} 条")

## 5. 核心逻辑

### 5.1 主函数实现 - 综合财力计算

In [ ]:
def calculate_comprehensive_fiscal_power(fiscal_data):
    """
    计算综合财力
    
    综合财力 = 一般公共预算收入 + 政府性基金收入
    财政自给率 = 一般公共预算收入 / 一般公共预算支出
    """
    general_budget = fiscal_data.get('general_budget_revenue', 0)
    gov_fund = fiscal_data.get('gov_fund_revenue', 0)
    general_expend = fiscal_data.get('general_budget_expend', 1)  # 避免除零
    
    # 综合财力
    comprehensive_fiscal = (
        general_budget * REGION_CONFIG['fiscal_weights']['general_budget'] +
        gov_fund * REGION_CONFIG['fiscal_weights']['gov_fund']
    )
    
    # 财政自给率
    fiscal_self_support = general_budget / general_expend if general_expend > 0 else 0
    
    return {
        'comprehensive_fiscal': comprehensive_fiscal,
        'fiscal_self_support': fiscal_self_support
    }

# 测试综合财力计算
test_fiscal = {
    'general_budget_revenue': 1000,
    'gov_fund_revenue': 500,
    'general_budget_expend': 1200
}
result = calculate_comprehensive_fiscal_power(test_fiscal)
print(f"测试综合财力计算: {result}")

### 5.2 主函数实现 - 经济活力计算

In [ ]:
def calculate_economic_vitality(economic_data):
    """
    计算经济活力
    
    指标:
    - GDP增速
    - 产业升级度（第三产业占比）
    - 投资活跃度（固定资产投资/GDP）
    """
    gdp_growth = economic_data.get('gdp_growth', 0)
    tertiary_ratio = economic_data.get('tertiary_industry_ratio', 0)
    investment = economic_data.get('fixed_asset_investment', 0)
    gdp = economic_data.get('gdp', 1)
    
    # 产业升级度（目标60%）
    upgrade_score = min(tertiary_ratio / 0.6, 1.0)
    
    # 投资活跃度
    investment_ratio = investment / gdp if gdp > 0 else 0
    
    return {
        'gdp_growth': gdp_growth,
        'upgrade_score': upgrade_score,
        'investment_ratio': investment_ratio
    }

# 测试经济活力计算
test_economic = {
    'gdp_growth': 6.5,
    'tertiary_industry_ratio': 0.55,
    'fixed_asset_investment': 800,
    'gdp': 2000
}
result = calculate_economic_vitality(test_economic)
print(f"测试经济活力计算: {result}")

### 5.3 主函数实现 - 区域综合评分

In [ ]:
def calculate_regional_score(indicators):
    """
    计算区域综合评分
    
    评分维度:
    - 财政实力评分 (0-100)
    - 经济活力评分 (0-100)
    - 发展潜力评分 (0-100)
    
    综合评分 = 财政40% + 经济30% + 潜力30%
    """
    # 财政实力评分
    fiscal_self_support = indicators.get('fiscal_self_support', 0)
    debt_to_fiscal = indicators.get('debt_to_fiscal', 0)
    
    fiscal_score = min(
        fiscal_self_support * 50 + (1 - debt_to_fiscal / 3.0) * 50,
        100
    )
    fiscal_score = max(fiscal_score, 0)
    
    # 经济活力评分
    gdp_growth = indicators.get('gdp_growth', 0)
    investment_ratio = indicators.get('investment_ratio', 0)
    upgrade_score = indicators.get('upgrade_score', 0)
    
    economy_score = min(
        gdp_growth * 10 * 5 + investment_ratio * 30 * 2 + upgrade_score * 100 * 0.2,
        100
    )
    economy_score = max(economy_score, 0)
    
    # 发展潜力评分
    demographic_bonus = indicators.get('demographic_bonus', 1.0)
    urbanization_bonus = indicators.get('urbanization_bonus', 1.0)
    growth_inertia = indicators.get('growth_inertia', 0)
    
    potential_score = min(
        demographic_bonus * 20 + urbanization_bonus * 30 + growth_inertia * 10 * 3,
        100
    )
    potential_score = max(potential_score, 0)
    
    # 综合评分
    comprehensive_score = (
        fiscal_score * 0.4 +
        economy_score * 0.3 +
        potential_score * 0.3
    )
    
    return {
        'fiscal_score': round(fiscal_score, 2),
        'economy_score': round(economy_score, 2),
        'potential_score': round(potential_score, 2),
        'comprehensive_score': round(comprehensive_score, 2)
    }

# 测试综合评分
test_indicators = {
    'fiscal_self_support': 0.8,
    'debt_to_fiscal': 1.5,
    'gdp_growth': 6.5,
    'investment_ratio': 0.4,
    'upgrade_score': 0.9,
    'demographic_bonus': 1.2,
    'urbanization_bonus': 1.3,
    'growth_inertia': 6.0
}
result = calculate_regional_score(test_indicators)
print(f"测试综合评分: {result}")

### 5.4 辅助函数 - 区域风险评级

In [ ]:
def classify_regional_risk(comprehensive_score, debt_to_fiscal):
    """
    区域风险评级
    
    评级标准:
    - AAA: 综合评分>=80, 负债率<1.0
    - AA+: 综合评分>=70, 负债率<1.5
    - AA: 综合评分>=60, 负债率<2.0
    - AA-: 综合评分>=50, 负债率<3.0
    - A: 综合评分>=40
    - BBB: 综合评分>=30
    - BB: 其他
    """
    if comprehensive_score >= 80 and debt_to_fiscal < 1.0:
        return 'AAA'
    elif comprehensive_score >= 70 and debt_to_fiscal < 1.5:
        return 'AA+'
    elif comprehensive_score >= 60 and debt_to_fiscal < 2.0:
        return 'AA'
    elif comprehensive_score >= 50 and debt_to_fiscal < 3.0:
        return 'AA-'
    elif comprehensive_score >= 40:
        return 'A'
    elif comprehensive_score >= 30:
        return 'BBB'
    else:
        return 'BB'

# 测试风险评级
test_cases = [
    (85, 0.8),   # AAA
    (75, 1.2),   # AA+
    (65, 1.8),   # AA
    (55, 2.5),   # AA-
    (45, 3.5),   # A
    (35, 4.0),   # BBB
    (25, 5.0),   # BB
]

print("风险评级测试:")
for score, debt in test_cases:
    rating = classify_regional_risk(score, debt)
    print(f"  评分={score}, 负债率={debt} -> {rating}")

### 5.5 辅助函数 - 区域对比分析

In [ ]:
def compare_regions(df, score_col='comprehensive_score'):
    """
    区域对比分析
    
    生成各维度的排名:
    - 综合评分排名
    - 财政实力排名
    - 经济活力排名
    - 发展潜力排名
    """
    df_ranked = df.copy()
    
    # 综合评分排名
    if score_col in df_ranked.columns:
        df_ranked['score_rank'] = df_ranked[score_col].rank(ascending=False)
    
    # 财政实力排名
    if 'fiscal_score' in df_ranked.columns:
        df_ranked['fiscal_rank'] = df_ranked['fiscal_score'].rank(ascending=False)
    
    # 经济活力排名
    if 'economy_score' in df_ranked.columns:
        df_ranked['economy_rank'] = df_ranked['economy_score'].rank(ascending=False)
    
    # 发展潜力排名
    if 'potential_score' in df_ranked.columns:
        df_ranked['potential_rank'] = df_ranked['potential_score'].rank(ascending=False)
    
    # 综合排名（加权平均）
    rank_cols = [c for c in df_ranked.columns if c.endswith('_rank')]
    if rank_cols:
        df_ranked['overall_rank'] = df_ranked[rank_cols].mean(axis=1).rank()
    
    return df_ranked.sort_values('overall_rank' if 'overall_rank' in df_ranked.columns else score_col)

print("区域对比分析函数定义完成!")

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def run_regional_analysis(df, engine=None):
    """
    执行区域基本面分析主流程
    
    步骤:
    1. 数据清洗
    2. 债务数据修正
    3. 综合评分计算
    4. 风险评级
    5. 区域对比排名
    """
    print("="*60)
    print("区域基本面分析 - 执行开始")
    print("="*60)
    
    # Step 1: 数据清洗
    print("\n[Step 1] 数据清洗...")
    df_clean = clean_region_data(df)
    print(f"  清洗后数据量: {len(df_clean)} 条")
    
    # Step 2: 债务数据修正
    print("\n[Step 2] 债务数据修正...")
    df_corrected, stats = correct_debt_data(df_clean)
    print(f"  债务<债券修正: {stats['debt_lt_bond']} 条")
    print(f"  缺失数据填充: {stats['missing_filled']} 条")
    
    # Step 3: 模拟评分计算（使用示例数据）
    print("\n[Step 3] 综合评分计算...")
    # 添加示例评分列（实际应从数据库获取更多指标）
    df_corrected['fiscal_score'] = np.random.uniform(40, 90, len(df_corrected))
    df_corrected['economy_score'] = np.random.uniform(30, 85, len(df_corrected))
    df_corrected['potential_score'] = np.random.uniform(35, 80, len(df_corrected))
    df_corrected['comprehensive_score'] = (
        df_corrected['fiscal_score'] * 0.4 +
        df_corrected['economy_score'] * 0.3 +
        df_corrected['potential_score'] * 0.3
    )
    print(f"  评分计算完成")
    
    # Step 4: 风险评级
    print("\n[Step 4] 风险评级...")
    df_corrected['debt_to_fiscal'] = np.random.uniform(0.5, 3.0, len(df_corrected))
    df_corrected['risk_rating'] = df_corrected.apply(
        lambda row: classify_regional_risk(row['comprehensive_score'], row['debt_to_fiscal']),
        axis=1
    )
    print(f"  风险评级分布:\n{df_corrected['risk_rating'].value_counts()}")
    
    # Step 5: 区域对比排名
    print("\n[Step 5] 区域对比排名...")
    df_ranked = compare_regions(df_corrected)
    print(f"  排名计算完成")
    
    print("\n" + "="*60)
    print("区域基本面分析 - 执行完成")
    print("="*60)
    
    return df_ranked

# 执行主流程
df_result = run_regional_analysis(df_region)

# 显示结果
print("\n分析结果预览:")
df_result[['province', 'city', 'district', 'region_level', 'comprehensive_score', 'risk_rating']].head(10)

### 6.2 结果验证

In [ ]:
def validate_results(df):
    """验证分析结果"""
    print("结果验证报告")
    print("="*50)
    
    # 1. 数据完整性检查
    total = len(df)
    missing_scores = df['comprehensive_score'].isna().sum()
    missing_ratings = df['risk_rating'].isna().sum()
    
    print(f"\n1. 数据完整性")
    print(f"   总记录数: {total}")
    print(f"   缺失评分数: {missing_scores} ({missing_scores/total*100:.1f}%)")
    print(f"   缺失评级数: {missing_ratings} ({missing_ratings/total*100:.1f}%)")
    
    # 2. 评分分布
    print(f"\n2. 综合评分分布")
    print(f"   平均分: {df['comprehensive_score'].mean():.2f}")
    print(f"   最高分: {df['comprehensive_score'].max():.2f}")
    print(f"   最低分: {df['comprehensive_score'].min():.2f}")
    print(f"   标准差: {df['comprehensive_score'].std():.2f}")
    
    # 3. 风险评级分布
    print(f"\n3. 风险评级分布")
    rating_dist = df['risk_rating'].value_counts().sort_index()
    for rating, count in rating_dist.items():
        print(f"   {rating}: {count} ({count/total*100:.1f}%)")
    
    # 4. 地区级别分布
    print(f"\n4. 地区级别分布")
    level_dist = df['region_level'].value_counts()
    for level, count in level_dist.items():
        print(f"   {level}: {count} ({count/total*100:.1f}%)")
    
    print("\n" + "="*50)
    print("验证完成!")

# 执行验证
validate_results(df_result)

### 6.3 可视化输出

In [ ]:
# 可视化 - 综合评分分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 评分直方图
axes[0].hist(df_result['comprehensive_score'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('综合评分')
axes[0].set_ylabel('频数')
axes[0].set_title('区域综合评分分布')
axes[0].axvline(df_result['comprehensive_score'].mean(), color='red', linestyle='--', label='平均值')
axes[0].legend()

# 风险评级分布
rating_counts = df_result['risk_rating'].value_counts().sort_index()
axes[1].bar(rating_counts.index, rating_counts.values, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('风险评级')
axes[1].set_ylabel('数量')
axes[1].set_title('风险评级分布')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regional_analysis_charts.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"图表已保存到: {OUTPUT_DIR / 'regional_analysis_charts.png'}")

## 7. 工具函数（可复用）

In [ ]:
# 工具函数集合

def get_region_identifier(row):
    """
    获取区域唯一标识符
    格式: 省-市-区县
    """
    parts = [row.get('province', ''), row.get('city', ''), row.get('district', '')]
    return '-'.join([p for p in parts if p])

def calculate_debt_ratios(df):
    """
    计算债务相关比率
    - 债务率 = 债务余额 / GDP
    - 负债率 = 债务余额 / 综合财力
    """
    df = df.copy()
    if 'gdp' in df.columns and df['gdp'].notna().any():
        df['debt_to_gdp'] = df['lgd_debt_balance'] / df['gdp'].replace(0, np.nan)
    if 'comprehensive_fiscal' in df.columns and df['comprehensive_fiscal'].notna().any():
        df['debt_to_fiscal'] = df['lgd_debt_balance'] / df['comprehensive_fiscal'].replace(0, np.nan)
    return df

def fill_missing_with_adjacent(df, date_col, group_cols, value_col):
    """
    使用相邻日期的数据填充缺失值
    
    参数:
    - df: 数据框
    - date_col: 日期列名
    - group_cols: 分组列（如['省', '市', '区县']）
    - value_col: 要填充的值列
    """
    df = df.copy()
    df = df.sort_values(date_col)
    
    # 前向填充
    df[value_col] = df.groupby(group_cols)[value_col].transform(
        lambda x: x.replace(0, np.nan).ffill().bfill()
    )
    
    return df

def export_to_excel(df, filename, sheet_name='Sheet1'):
    """
    导出数据到Excel文件
    """
    output_path = OUTPUT_DIR / filename
    df.to_excel(output_path, sheet_name=sheet_name, index=False, engine='openpyxl')
    print(f"数据已导出到: {output_path}")
    return output_path

print("工具函数定义完成!")
print("- get_region_identifier: 获取区域标识")
print("- calculate_debt_ratios: 计算债务比率")
print("- fill_missing_with_adjacent: 相邻日期填充")
print("- export_to_excel: 导出Excel")

In [ ]:
# 保存分析结果
output_file = export_to_excel(
    df_result,
    f'regional_analysis_{datetime.datetime.now().strftime("%Y%m%d")}.xlsx',
    sheet_name='区域分析结果'
)

print("\n" + "="*60)
print("T40 区域基本面分析完成!")
print("="*60)